In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ===== 0. 设备：有 GPU 用 GPU =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# ===== 1. 数据 =====
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

# ===== 2. 模型 =====
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = MLP().to(device)          # 模型搬到 GPU

# ===== 3. 损失函数 + 优化器 =====
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# ===== 4. 训练（五步循环）=====
for epoch in range(5):
    model.train()                 # 切到训练模式
    for images, labels in train_loader:
        images = images.view(images.size(0), -1).to(device)  # 展平 + 搬到 GPU
        labels = labels.to(device)                            # 标签也搬到 GPU

        outputs = model(images)              # ① 前向
        loss = criterion(outputs, labels)    # ② 算损失
        optimizer.zero_grad()                # ③ 清零梯度
        loss.backward()                      # ④ 反向求梯度
        optimizer.step()                     # ⑤ 更新参数

    print(f"Epoch {epoch+1}, loss={loss.item():.4f}")

# ===== 5. 评估 =====
model.eval()                      # 切到评估模式
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(images.size(0), -1).to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

使用设备: cuda


100.0%
100.0%
100.0%
100.0%


Epoch 1, loss=0.2725
Epoch 2, loss=0.0707
Epoch 3, loss=0.2197
Epoch 4, loss=0.0224
Epoch 5, loss=0.2414
Test Accuracy: 96.61%
